# banao-tech MuseTalk on a free Colab GPU — timing test

This runs the **exact `banao-tech/musetalk-avatar` recipe** (the one that worked for you), but on Colab's **GPU** instead of the Space's CPU, so you get the real speed. It:
1. clones the banao Space (its code + configs + the working command),
2. installs the same pinned deps, patched to **GPU** (torch 2.0.1+cu118, mmcv 2.0.1),
3. downloads the weights (incl. whisper-tiny, like banao),
4. runs banao's exact inference on your portrait + voice and **prints the time**.

**First:** Runtime -> Change runtime type -> **GPU (T4)**.

**Install note:** cell 3a downgrades torch, so Colab may ask to **restart the session**. Do it, then run 3b (not 3a again). Cell 3b verifies `mmpose` imported with CUDA.

In [ ]:
# 1. Confirm a GPU is attached
!nvidia-smi

In [ ]:
# 2. System libs + clone the banao Space (it IS a git repo)
!apt-get -y -qq install ffmpeg libgl1 libglib2.0-0 libsm6 libxext6 cmake build-essential git-lfs
%cd /content
!git clone https://huggingface.co/spaces/banao-tech/musetalk-avatar
%cd /content/musetalk-avatar

In [ ]:
# 3a. Pin torch 2.0.1 + CUDA 11.8 FIRST (the GPU twin of banao's CPU pins). mmcv 2.0.1
#     only has wheels for this torch, so this is what makes mmpose install.
!pip install -q numpy==1.26.4
!pip install -q torch==2.0.1 torchvision==0.15.2 torchaudio==2.0.2 --index-url https://download.pytorch.org/whl/cu118
print("\n>>> If Colab prompts to restart: Runtime > Restart session, then run 3b (NOT 3a).")

In [ ]:
# 3b. banao's deps, patched to GPU: install mmcv from the cu118 index, and the rest of
#     banao's requirements minus the CPU torch/mmcv/numpy lines.
%cd /content/musetalk-avatar
!pip install -q --upgrade pip "setuptools==68.2.2" wheel cython poetry-core
!pip install -q --no-build-isolation chumpy==0.70
!pip install -q mmcv==2.0.1 -f https://download.openmmlab.com/mmcv/dist/cu118/torch2.0.0/index.html
# drop the CPU index line and the torch/mmcv/numpy pins from banao's requirements
!grep -vE '^-f |^torch==|^torchvision==|^torchaudio==|^mmcv==|^numpy==' requirements.txt > reqs_gpu.txt
!pip install -q -r reqs_gpu.txt

# Verify the fragile imports NOW (this is where the official-repo run died)
import importlib
ok = True
for m in ["torch", "mmcv", "mmdet", "mmpose"]:
    try:
        mod = importlib.import_module(m)
        extra = f" cuda={mod.cuda.is_available()}" if m == "torch" else ""
        print("OK  ", m, getattr(mod, "__version__", "?"), extra)
    except Exception as e:
        ok = False
        print("FAIL", m, "->", e)
print("\nAll good — continue." if ok else
      "\n>>> Still failing: Runtime > Restart session, then run 3b again.")

In [ ]:
# 4. Download weights into the repo's models/ dir (incl. whisper-tiny, like banao)
import os
os.environ["MODEL_DIR"] = "/content/musetalk-avatar/models"
!bash download_weights.sh

## Your character — upload portrait + voice
banao feeds the **image directly** to MuseTalk (no image->video step needed), which is why it worked. Upload `poppy.jpg` and `poppy.wav`.

In [ ]:
# Upload the portrait and the audio in two separate dialogs
from google.colab import files

print("Please upload your portrait image:")
img_upload = files.upload()
img_names = list(img_upload.keys())
img = next((n for n in img_names if n.lower().endswith(('.jpg', '.jpeg', '.png'))), None)

print("\nPlease upload your audio file:")
wav_upload = files.upload()
wav_names = list(wav_upload.keys())
wav = next((n for n in wav_names if n.lower().endswith(('.wav', '.mp3', '.m4a'))), None)

print("portrait:", img, "| audio:", wav)
assert img and wav, "Upload one image AND one audio file."

import os
os.makedirs("data/custom", exist_ok=True)
os.replace(img, f"data/custom/{img}")
os.replace(wav, f"data/custom/{wav}")

In [ ]:
# Run banao's EXACT inference command on your files, with timing + full error capture.
import os, time, glob, base64, subprocess, soundfile as sf
from IPython.display import HTML

MODEL_DIR = "/content/musetalk-avatar/models"
os.makedirs("configs/inference", exist_ok=True)
yaml_text = f'''space_avatar:
  video_path: "data/custom/{img}"
  audio_path: "data/custom/{wav}"
  bbox_shift: 0
'''
open("configs/inference/colab_test.yaml", "w").write(yaml_text)
print(yaml_text)

info = sf.info(f"data/custom/{wav}")
audio_sec = info.frames / info.samplerate

cmd = [
    "python", "-m", "scripts.inference",
    "--inference_config", "configs/inference/colab_test.yaml",
    "--result_dir", "results/colab",
    "--unet_model_path", f"{MODEL_DIR}/musetalkV15/unet.pth",
    "--unet_config", f"{MODEL_DIR}/musetalkV15/musetalk.json",
    "--version", "v15", "--fps", "15", "--batch_size", "4", "--bbox_shift", "0",
    "--parsing_mode", "jaw", "--left_cheek_width", "90", "--right_cheek_width", "90",
]
env = os.environ.copy(); env["MODEL_DIR"] = MODEL_DIR
t0 = time.time()
r = subprocess.run(cmd, env=env, capture_output=True, text=True)
dt = time.time() - t0

print(f"\n===== TIMING on this Colab GPU =====")
print(f"exit code        : {r.returncode}")
print(f"clip length      : {audio_sec:5.1f} s of audio")
print(f"generation time  : {dt:5.1f} s")
if audio_sec:
    print(f"realtime factor  : {dt/audio_sec:5.2f}x  (only meaningful if exit code is 0)")

vids = sorted(glob.glob("results/**/*.mp4", recursive=True), key=os.path.getmtime)
if r.returncode == 0 and vids:
    print("Showing:", vids[-1])
    b64 = base64.b64encode(open(vids[-1], 'rb').read()).decode()
    display(HTML(f'<video width=480 controls autoplay loop src="data:video/mp4;base64,{b64}"></video>'))
else:
    print("\n===== INFERENCE FAILED — last 4000 chars of the log =====")
    tail = (r.stdout or "") + "\n---- STDERR ----\n" + (r.stderr or "")
    print(tail[-4000:])

### Read the result
- **generation time / realtime factor** is the number we care about. The banao *Space* was CPU (~10 min/5s); this T4 should be a small fraction of that, and a g5 faster still.
- Judge lip-sync quality too. banao runs low-res / fps=15 for speed; higher res is possible on the g5.
- If the realtime factor on the T4 is roughly <=1-2x, MuseTalk on your g5 is easily turn-based-ready (and realtime mode gets you toward live). If it's much worse, we look at a lighter model.